In [1]:
# 05_evaluation.ipynb
# Цель:
# 1) показать на toy example, как оценивается retrieval
# 2) посчитать метрики для bm25, dense и random baseline
# 3) сохранить per_query_results.csv и metrics_summary.csv

# Что мы оцениваем

У нас есть:
- вопрос (`question`)
- правильный релевантный чанк (`relevant_chunk_id`)
- ranking-список чанков, который вернул метод retrieval

Наша задача:
- проверить, попал ли правильный чанк в top-10
- если попал, то на каком месте
- сравнить разные методы retrieval

# Метрики

Обозначим через `r_i` ранг первого релевантного чанка для i-го вопроса.

Если релевантный чанк не попал в top-10, то считаем:
- для `Hit@10` и `Recall@10`: попадания нет
- для `MRR`: вклад равен 0
- для `Mean First Relevant Rank`: присваиваем значение 11

---

## 1. Hit@10

Показывает, попал ли релевантный чанк в первые 10 результатов.

Формула:

$Hit@10 = (1 / N) * Σ I(r_i <= 10)$

где I(...) = 1, если условие выполнено, иначе 0.

---

## 2. Recall@10

В общем случае Recall@10 показывает, какую долю релевантных документов мы нашли в top-10.

Но в этом проекте на каждый вопрос есть только **один** релевантный чанк.

Поэтому здесь:

Recall@10 = Hit@10

---

## 3. MRR

Mean Reciprocal Rank — среднее значение величины 1 / rank.

Формула:

$MRR = (1 / N) * Σ (1 / r_i)$, если релевантный чанк найден,
иначе вклад равен 0.

---

## 4. Mean First Relevant Rank

Средний ранг первого релевантного чанка.

В этом проекте договоримся:
- если релевантный чанк найден на позиции 1..10, берём этот ранг
- если не найден в top-10, ставим 11


Более подробное пояснение метрик можете найти самостоятельно

In [2]:
# Условный пример

import numpy as np
import pandas as pd

In [4]:
toy_gold_df = pd.DataFrame({
    "query_id": ["q001", "q002", "q003"],
    "question": [
        "Какая выручка компании за 2024 год?",
        "Какая чистая прибыль компании?",
        "Какие капитальные затраты были в 2024 году?"
    ],
    "relevant_chunk_id": ["d3", "d7", "d2"]
})

toy_gold_df

,query_id,question,relevant_chunk_id
0,q001,Какая выручка компании за 2024 год?,d3
1,q002,Какая чистая прибыль компании?,d7
2,q003,Какие капитальные затраты были в 2024 году?,d2


In [5]:
toy_rankings_df = pd.DataFrame({
    "query_id": [
        "q001","q001","q001","q001","q001",
        "q002","q002","q002","q002","q002",
        "q003","q003","q003","q003","q003",
        
        "q001","q001","q001","q001","q001",
        "q002","q002","q002","q002","q002",
        "q003","q003","q003","q003","q003",
        
        "q001","q001","q001","q001","q001",
        "q002","q002","q002","q002","q002",
        "q003","q003","q003","q003","q003",
    ],
    "method": (
        ["bm25"] * 15 +
        ["dense"] * 15 +
        ["random"] * 15
    ),
    "rank": [1,2,3,4,5] * 9,
    "chunk_id": [
        # bm25
        "d3","d1","d8","d5","d4",
        "d4","d1","d7","d8","d2",
        "d9","d8","d5","d1","d4",

        # dense
        "d1","d3","d5","d2","d4",
        "d7","d6","d5","d1","d3",
        "d2","d4","d6","d9","d8",

        # random
        "d8","d4","d1","d6","d3",
        "d5","d2","d1","d8","d9",
        "d4","d6","d9","d5","d1",
    ],
})
toy_rankings_df

,query_id,method,rank,chunk_id
0,q001,bm25,1,d3
1,q001,bm25,2,d1
2,q001,bm25,3,d8
3,q001,bm25,4,d5
4,q001,bm25,5,d4
5,q002,bm25,1,d4
6,q002,bm25,2,d1
7,q002,bm25,3,d7
8,q002,bm25,4,d8
9,q002,bm25,5,d2


# Что видно глазами

Для toy example:

## BM25
- q001: правильный чанк `d3` стоит на месте 1
- q002: правильный чанк `d7` стоит на месте 3
- q003: правильного чанка `d2` в списке нет

## Dense
- q001: правильный чанк `d3` стоит на месте 2
- q002: правильный чанк `d7` стоит на месте 1
- q003: правильный чанк `d2` стоит на месте 1

## Random
- q001: правильный чанк `d3` стоит на месте 5
- q002: правильного чанка `d7` в списке нет
- q003: правильного чанка `d2` в списке нет

In [ ]:
def get_first_relevant_rank(ranking_df_for_one_query, relevant_chunk_id, max_rank=10):
    """
    Возвращает ранг релевантного чанка для одного вопроса.

    Если relevant_chunk_id найден в ranking_df_for_one_query,
    возвращается его rank.

    Если relevant_chunk_id не найден, возвращается max_rank + 1.
    В нашем проекте при max_rank=10 это значит 11.
    """
    matched = ranking_df_for_one_query[
        ranking_df_for_one_query["chunk_id"] == relevant_chunk_id
    ]

    if len(matched) == 0:
        return max_rank + 1

    return int(matched["rank"].min())

# Проверим
toy_q1_bm25 = toy_rankings_df[
    (toy_rankings_df["query_id"] == "q001") &
    (toy_rankings_df["method"] == "bm25")
]

print(get_first_relevant_rank(toy_q1_bm25, "d3", max_rank=5))

1


In [8]:
def build_per_query_results(gold_df, rankings_df, max_rank=10):
    """
    Строит таблицу per-query результатов.

    Что делает функция:
    1. Берёт все методы из rankings_df
    2. Для каждого метода проходит по всем вопросам из gold_df
    3. Для каждого вопроса находит правильный relevant_chunk_id
    4. Берёт ranking только для этого query_id и method
    5. Находит ранг первого релевантного чанка
    6. Собирает итоговую таблицу

    Возвращает DataFrame с колонками:
    - query_id
    - method
    - relevant_chunk_id
    - first_relevant_rank
    """
    rows = []

    methods = rankings_df["method"].dropna().unique().tolist()

    for method in methods:
        method_rankings = rankings_df[rankings_df["method"] == method].copy()

        for _, gold_row in gold_df.iterrows():
            query_id = gold_row["query_id"]
            relevant_chunk_id = gold_row["relevant_chunk_id"]

            ranking_for_one_query = method_rankings[
                method_rankings["query_id"] == query_id
            ].copy()

            first_relevant_rank = get_first_relevant_rank(
                ranking_df_for_one_query=ranking_for_one_query,
                relevant_chunk_id=relevant_chunk_id,
                max_rank=max_rank,
            )

            rows.append({
                "query_id": query_id,
                "method": method,
                "relevant_chunk_id": relevant_chunk_id,
                "first_relevant_rank": first_relevant_rank,
            })

    per_query_df = pd.DataFrame(rows)
    per_query_df = per_query_df.sort_values(
        ["method", "query_id"]
    ).reset_index(drop=True)

    return per_query_df

In [9]:
toy_per_query_df = build_per_query_results(
    gold_df=toy_gold_df,
    rankings_df=toy_rankings_df,
    max_rank=5,
)

toy_per_query_df

,query_id,method,relevant_chunk_id,first_relevant_rank
0,q001,bm25,d3,1
1,q002,bm25,d7,3
2,q003,bm25,d2,6
3,q001,dense,d3,2
4,q002,dense,d7,1
5,q003,dense,d2,1
6,q001,random,d3,5
7,q002,random,d7,6
8,q003,random,d2,6


In [10]:
def compute_hit_at_k(per_query_df, k=10):
    """
    Считает Hit@K.

    Что нужно сделать:
    - для каждой строки проверить, что first_relevant_rank <= k
    - перевести это в 0/1
    - вернуть среднее значение
    """
    hits = (per_query_df["first_relevant_rank"] <= k).astype(int)
    return hits.mean()

In [11]:
for method in toy_per_query_df["method"].unique():
    part = toy_per_query_df[toy_per_query_df["method"] == method]
    print(method, "Hit@5 =", compute_hit_at_k(part, k=5))

bm25 Hit@5 = 0.6666666666666666
dense Hit@5 = 1.0
random Hit@5 = 0.3333333333333333


# Что нужно реализовать самим

На основе `first_relevant_rank` нужно самостоятельно посчитать ещё 3 метрики:

## 1. Recall@10
В этом проекте Recall@10 должен совпасть с Hit@10,
потому что на каждый вопрос есть только один релевантный чанк.

## 2. MRR
Для каждого вопроса:
- если `first_relevant_rank <= 10`, вклад = `1 / first_relevant_rank`
- иначе вклад = 0

Потом нужно усреднить по всем вопросам.

## 3. Mean First Relevant Rank
Для каждого вопроса берём `first_relevant_rank`
(включая значение 11, если релевантный чанк не найден),
потом усредняем.

In [ ]:
GOLD_PATH = "gold_final.csv"
CHUNKS_PATH = "chunks_final.csv"

BM25_PATH = "bm25_rankings.csv"
DENSE_PATH = "dense_rankings.csv"
RANDOM_PATH = "random_rankings.csv"

PER_QUERY_OUTPUT_PATH = "per_query_results.csv"
METRICS_OUTPUT_PATH = "metrics_summary.csv"

TOP_K = 10
RANDOM_SEED = 42

In [ ]:
gold_df = pd.read_csv(GOLD_PATH)
chunks_df = pd.read_csv(CHUNKS_PATH)
bm25_df = pd.read_csv(BM25_PATH)
dense_df = pd.read_csv(DENSE_PATH)

print("gold_df:", gold_df.shape)
print("chunks_df:", chunks_df.shape)
print("bm25_df:", bm25_df.shape)
print("dense_df:", dense_df.shape)

In [ ]:
required_gold_cols = ["query_id", "relevant_chunk_id"]
required_rank_cols = ["query_id", "method", "rank", "chunk_id", "score"]
required_chunk_cols = ["chunk_id"]

missing_gold_cols = [c for c in required_gold_cols if c not in gold_df.columns]
missing_chunk_cols = [c for c in required_chunk_cols if c not in chunks_df.columns]
missing_bm25_cols = [c for c in required_rank_cols if c not in bm25_df.columns]
missing_dense_cols = [c for c in required_rank_cols if c not in dense_df.columns]

assert not missing_gold_cols, f"В gold_final.csv не хватает колонок: {missing_gold_cols}"
assert not missing_chunk_cols, f"В chunks_final.csv не хватает колонок: {missing_chunk_cols}"
assert not missing_bm25_cols, f"В bm25_rankings.csv не хватает колонок: {missing_bm25_cols}"
assert not missing_dense_cols, f"В dense_rankings.csv не хватает колонок: {missing_dense_cols}"

print("Проверка входных файлов пройдена")

In [ ]:
# Сгенерируем random baseline
rng = np.random.default_rng(RANDOM_SEED)

all_chunk_ids = chunks_df["chunk_id"].astype(str).tolist()
random_rows = []

for query_id in gold_df["query_id"].astype(str).tolist():
    sampled_chunk_ids = rng.permutation(all_chunk_ids)[:TOP_K]

    for rank, chunk_id in enumerate(sampled_chunk_ids, start=1):
        random_rows.append({
            "query_id": query_id,
            "method": "random",
            "rank": rank,
            "chunk_id": chunk_id,
            "score": 0.0,
        })

random_df = pd.DataFrame(random_rows)
random_df.to_csv(RANDOM_PATH, index=False, encoding="utf-8-sig")

print("random_df:", random_df.shape)
display(random_df.head(20))

In [ ]:
all_rankings_df = pd.concat(
    [bm25_df, dense_df, random_df],
    ignore_index=True
)

print(all_rankings_df.shape)
display(all_rankings_df.head(20))

In [ ]:
per_query_results_df = build_per_query_results(
    gold_df=gold_df,
    rankings_df=all_rankings_df,
    max_rank=TOP_K,
)

print("per_query_results_df:", per_query_results_df.shape)
display(per_query_results_df.head(20))

In [ ]:
required_per_query_cols = [
    "query_id",
    "method",
    "relevant_chunk_id",
    "first_relevant_rank",
]

missing_per_query_cols = [c for c in required_per_query_cols if c not in per_query_results_df.columns]
assert not missing_per_query_cols, f"Не хватает колонок: {missing_per_query_cols}"

assert per_query_results_df["query_id"].notna().all()
assert per_query_results_df["method"].notna().all()
assert per_query_results_df["relevant_chunk_id"].notna().all()
assert per_query_results_df["first_relevant_rank"].notna().all()

print("Проверка per_query_results_df пройдена")

# Итоговое задание

Используя `per_query_results_df`, посчитайте для каждого метода 4 метрики:

- Hit@10
- Recall@10
- MRR
- Mean First Relevant Rank

Нужно получить итоговую таблицу `metrics_summary_df` с колонками:

- method
- hit_at_10
- recall_at_10
- mrr
- mean_first_relevant_rank
- n_queries

Подсказка:
- группируйте `per_query_results_df` по `method`
- считайте каждую метрику отдельно
- потом соберите одну итоговую таблицу

In [7]:
def build_metrics_summary(per_query_results_df):
    """
    Строит итоговую таблицу метрик по методам.

    Что нужно сделать:
    1. пройти по всем методам
    2. для каждого метода взять соответствующую часть per_query_results_df
    3. посчитать:
       - hit_at_10
       - recall_at_10
       - mrr
       - mean_first_relevant_rank
       - n_queries
    4. собрать результат в DataFrame
    """
    # TODO: реализовать
    raise NotImplementedError

In [ ]:
metrics_summary_df = build_metrics_summary(per_query_results_df)
metrics_summary_df

In [ ]:
assert "method" in metrics_summary_df.columns
assert "hit_at_10" in metrics_summary_df.columns
assert "recall_at_10" in metrics_summary_df.columns
assert "mrr" in metrics_summary_df.columns
assert "mean_first_relevant_rank" in metrics_summary_df.columns
assert "n_queries" in metrics_summary_df.columns

print(metrics_summary_df)

# В этом проекте Recall@10 должен совпадать с Hit@10
assert np.allclose(
    metrics_summary_df["hit_at_10"].values,
    metrics_summary_df["recall_at_10"].values
), "Recall@10 должен совпадать с Hit@10 при одном релевантном чанке на вопрос"

print("Sanity check пройден")

In [ ]:
per_query_results_df.to_csv(PER_QUERY_OUTPUT_PATH, index=False, encoding="utf-8-sig")
metrics_summary_df.to_csv(METRICS_OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Сохранён файл: {PER_QUERY_OUTPUT_PATH}")
print(f"Сохранён файл: {METRICS_OUTPUT_PATH}")